In [30]:
import base64
import operator
import subprocess
import textwrap
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send, interrupt
from openai import OpenAI

memory = InMemorySaver()

llm = init_chat_model("openai:gpt-4o-mini")


class State(TypedDict):
    video_file: str
    audio_file: str
    transcription: str
    summaries: Annotated[list[str], operator.add]
    thumbnail_prompts: Annotated[list[str], operator.add]
    thumbnail_sketches: Annotated[list[str], operator.add]
    final_summary: str
    user_feedback: str
    chosen_prompt: str

In [31]:
def extract_audio(state: State):
    output_file = state["video_file"].replace("mp4", "mp3")
    command = [
        "ffmpeg",
        "-i",
        state["video_file"],
        "-filter:a",
        "atempo=2.0",
        "-y",
        output_file,
    ]
    subprocess.run(command, check=True)
    return {
        "audio_file": output_file,
    }


def transcribe_audio(state: State):
    client = OpenAI()
    with open(state["audio_file"], "rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            model="whisper-1",
            response_format="text",
            file=audio_file,
            language="en",
            prompt="cloudeflare, ai, agent",
        )
        return {
            "transcription": transcription,
        }


def dispatch_summarizers(state: State):
    transcription = state["transcription"]
    chunks = []
    for i, chunk in enumerate(textwrap.wrap(transcription, 500)):
        chunks.append(
            {
                "id": i + 1,
                "chunk": chunk,
            }
        )
    return [Send("summarize_chunk", chunk) for chunk in chunks]


def summarize_chunk(chunk):
    chunk_id = chunk["id"]
    chunk = chunk["chunk"]

    response = llm.invoke(
        f"""
        Please summarize the following text.
        
        Text: {chunk}
        """
    )
    summary = f"[Chunk {chunk_id}] {response.content}"
    return {"summaries": [summary]}

    # print(f"Summarizing chunk id: {chunk_id} chunk: {chunk[:100]}\n\n====\n\n")


def mega_summary(state: State):
    all_summaries = "\n".join(state["summaries"])

    prompt = f"""
        You are given multiple summaries of different chunks from a video transcription.
        
        Please create a comprehensive final summary that combines all the key points.
        
        The final summary must be written in Korean.
        
        Individual summaries: 
        
        {all_summaries}
        
    """
    response = llm.invoke(prompt)

    return {
        "final_summary": response.content,
    }


def dispatch_artists(state: State):
    return [
        Send(
            "generate_thumbnail",
            {
                "id": i,
                "summary": state["final_summary"],
            },
        )
        for i in range(3)
    ]


def generate_thumbnail(args):
    concept_id = args["id"]
    summary = args["summary"]

    prompt = f"""
    Based on this video summary, create a detailed visual prompt for a YouTube thumbnail.

    Create a detailed prompt for generating a thumbnail image that would attract viewers. Include:
        - Main visual elements
        - Color scheme
        - Text overlay suggestions
        - Overall composition
    
    Summary: {summary}
    """

    response = llm.invoke(prompt)

    thumbnail_prompt = response.content

    client = OpenAI()

    result = client.images.generate(
        model="gpt-image-1.5",
        prompt=thumbnail_prompt,
        quality="low",
        moderation="low",
        size="auto",
    )

    image_bytes = base64.b64decode(result.data[0].b64_json)

    filename = f"thumbnail_{concept_id}.jpg"

    with open(filename, "wb") as file:
        file.write(image_bytes)

    return {
        "thumbnail_prompts": [thumbnail_prompt],
        "thumbnail_sketches": [filename],
    }


def human_feedback(state: State):
    answer = interrupt(
        {
            "chosen_thumbnail": "Which thumbnail do you like most?",
            "feedback": "Provide any feedback or changes you'd like for the final thumbnail.",
        }
    )
    user_feedback = answer["user_feedback"]
    chosen_prompt = answer["chosen_prompt"]

    return {
        "user_feedback": user_feedback,
        "chosen_prompt": state["thumbnail_prompts"][chosen_prompt - 1],
    }


def generate_hd_thumbnail(state: State):
    chosen_prompt = state["chosen_prompt"]
    user_feedback = state["user_feedback"]

    prompt = f"""
    You are a professional YouTube thumbnail designer. Take this original thumbnail prompt and create an enhanced version that incorporates the user's specific feedback.

    ORIGINAL PROMPT:
    {chosen_prompt}

    USER FEEDBACK TO INCORPORATE:
    {user_feedback}

    Create an enhanced prompt that:
        1. Maintains the core concept from the original prompt
        2. Specifically addresses and implements the user's feedback requests
        3. Adds professional YouTube thumbnail specifications:
            - High contrast and bold visual elements
            - Clear focal points that draw the eye
            - Professional lighting and composition
            - Optimal text placement and readability with generous padding from edges
            - Colors that pop and grab attention
            - Elements that work well at small thumbnail sizes
            - IMPORTANT: Always ensure adequate white space/padding between any text and the image borders
    """

    response = llm.invoke(prompt)

    final_thumbnail_prompt = response.content

    client = OpenAI()

    result = client.images.generate(
        model="gpt-image-1.5",
        prompt=final_thumbnail_prompt,
        quality="high",
        moderation="low",
        size="auto",
    )

    image_bytes = base64.b64decode(result.data[0].b64_json)

    with open("thumbnail_final.jpg", "wb") as file:
        file.write(image_bytes)

In [32]:
graph_builder = StateGraph(State)

graph_builder.add_node("extract_audio", extract_audio)
graph_builder.add_node("transcribe_audio", transcribe_audio)
graph_builder.add_node("summarize_chunk", summarize_chunk)
graph_builder.add_node("mega_summary", mega_summary)
graph_builder.add_node("generate_thumbnail", generate_thumbnail)
graph_builder.add_node("human_feedback", human_feedback)
graph_builder.add_node("generate_hd_thumbnail", generate_hd_thumbnail)

graph_builder.add_edge(START, "extract_audio")
graph_builder.add_edge("extract_audio", "transcribe_audio")
graph_builder.add_conditional_edges(
    "transcribe_audio",
    dispatch_summarizers,
    ["summarize_chunk"],
)
graph_builder.add_edge("summarize_chunk", "mega_summary")
graph_builder.add_conditional_edges(
    "mega_summary",
    dispatch_artists,
    ["generate_thumbnail"],
)
graph_builder.add_edge("generate_thumbnail", "human_feedback")
graph_builder.add_edge("human_feedback", "generate_hd_thumbnail")
graph_builder.add_edge("generate_hd_thumbnail", END)

graph = graph_builder.compile(checkpointer=memory)

In [33]:
config = {
    "configurable": {
        "thread_id": "1",
    }
}

In [34]:
graph.invoke(
    {"video_file": "nomadcoder_youtube.mp4"},
    config=config,
)

ffmpeg version 8.1.2 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 21.0.0 (clang-2100.0.123.102)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.1.2_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60. 26.102 / 60. 26.102
  libavcodec     62. 28.102 / 62. 28.102
  libavformat    62. 12.102 / 62. 12.102
  libavdevice    62.  3.102 / 62.  3.102
  libavfilter    11. 14.102 / 11. 14.102
  libswscale      9.  5.102 /  9.  5.102
  libswresample   6.  3.102 /  6.  3.102
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from 'nomadcoder_youtube.mp4':
  Metadata:
    major_brand     : mp42
    minor_version   : 0
    compatible_brands: isommp42
    encoder         :

{'video_file': 'nomadcoder_youtube.mp4',
 'audio_file': 'nomadcoder_youtube.mp3',
 'transcription': "I just built the most advanced AI agent of my life It has tons of features, but I'm gonna show you the 5 most mind-blowing ones I open nomadclaw.xyz since the agent works on the web I tell the agent to go to nomadcallers.com, and a browser pops up But this browser is not on my computer, it is running on Cloudsor And both me and the agent can click on it, we share the same Chrome tab Now I ask the agent to take a screenshot, and boom, there it is A picture of the page we're both looking at, insane I write an email to bot at nomadclaw.xyz I tell it to remind me to work out in 30 seconds I send it, and I wait, I get a reply And in the chat, there it is, the agent sends a reminder And after 30 seconds, it reminded me Insane, the agent can receive email Now here's where things get incredibly cool We know all agents have tools, and agents call those tools on the server This agent, though, it 

In [39]:
snapshot = graph.get_state(config=config)

snapshot.interrupts


()

In [38]:
response = {
    "user_feedback": "사람을 지금처럼 만화적 느낌이 아니라 실제 사람 느낌으로 바꿔주고 너무 펑키한 느낌이야 조금 진중한 느낌으로 만들었으면 해",
    "chosen_prompt": 3,
}

graph.invoke(
    Command(resume=response),
    config=config,
)

{'video_file': 'nomadcoder_youtube.mp4',
 'audio_file': 'nomadcoder_youtube.mp3',
 'transcription': "I just built the most advanced AI agent of my life It has tons of features, but I'm gonna show you the 5 most mind-blowing ones I open nomadclaw.xyz since the agent works on the web I tell the agent to go to nomadcallers.com, and a browser pops up But this browser is not on my computer, it is running on Cloudsor And both me and the agent can click on it, we share the same Chrome tab Now I ask the agent to take a screenshot, and boom, there it is A picture of the page we're both looking at, insane I write an email to bot at nomadclaw.xyz I tell it to remind me to work out in 30 seconds I send it, and I wait, I get a reply And in the chat, there it is, the agent sends a reminder And after 30 seconds, it reminded me Insane, the agent can receive email Now here's where things get incredibly cool We know all agents have tools, and agents call those tools on the server This agent, though, it 